<a href="https://colab.research.google.com/github/vzdushzgsw/ev_traction_motor_thermal_digital_twin/blob/main/ev_thermal_mangement_model_execution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import numpy as np
import pandas as pd
import time

class TractionMotorDigitalTwin:
    def __init__(self, baseline_temp=30.0):
        print("Initializing Virtual Asset: EV Traction Motor Digital Twin...")
        self.winding_resistance_ohm = 0.05
        self.thermal_capacitance = 1200.0
        self.cooling_efficiency_factor = 45.0

        self.stator_temperature_celsius = baseline_temp
        self.cumulative_thermal_stress = 0.0

    def update_thermal_state(self, current_amps, coolant_flow_lmin, ambient_temp=25.0, dt_seconds=1.0):
        # I²R Copper Losses
        heat_generated = (current_amps ** 2) * self.winding_resistance_ohm

        # Convection Cooling
        effective_cooling = self.cooling_efficiency_factor * (coolant_flow_lmin / 5.0)
        heat_dissipated = effective_cooling * (self.stator_temperature_celsius - ambient_temp)

        # Thermodynamics: Net Heat updates Temperature
        net_heat = heat_generated - heat_dissipated
        self.stator_temperature_celsius += (net_heat / self.thermal_capacitance) * dt_seconds

    def get_health_telemetry(self):
        return {"Stator_Temp_C": round(self.stator_temperature_celsius, 2)}

# SIMULATION RUNTIME & DATA EXPORT
if __name__ == "__main__":
    print("Generating comprehensive dataset (Safe + Critical)...")
    telemetry_clipboard = []

    # We test a wide range to force some cases to trigger the '1' (Critical)
    for amps in np.linspace(100, 700, 25):   # High amps increase heat
        for flow in np.linspace(0.5, 5.0, 10): # Low flow prevents cooling

            # Reset motor for every scenario
            twin = TractionMotorDigitalTwin(baseline_temp=30.0)

            # Simulate 60 seconds of operation
            for _ in range(60):
                twin.update_thermal_state(amps, flow, dt_seconds=1.0)

            temp = twin.stator_temperature_celsius
            is_critical = 1 if temp > 140.0 else 0

            # Append this "outcome" to our data list
            telemetry_clipboard.append({
                'Stator_Current_Amps': amps,
                'Coolant_Flow_Lmin': flow,
                'Stator_Temp_C': temp,
                'Turtle_Trigger': is_critical
            })

    # Save and confirm
    df = pd.DataFrame(telemetry_clipboard)
    df.to_csv('synthetic_motor_data.csv', index=False)

    # Verification: Print how many critical cases we found
    print(f"Data generated! Critical events found: {df['Turtle_Trigger'].sum()}")

Generating comprehensive dataset (Safe + Critical)...
Initializing Virtual Asset: EV Traction Motor Digital Twin...
Initializing Virtual Asset: EV Traction Motor Digital Twin...
Initializing Virtual Asset: EV Traction Motor Digital Twin...
Initializing Virtual Asset: EV Traction Motor Digital Twin...
Initializing Virtual Asset: EV Traction Motor Digital Twin...
Initializing Virtual Asset: EV Traction Motor Digital Twin...
Initializing Virtual Asset: EV Traction Motor Digital Twin...
Initializing Virtual Asset: EV Traction Motor Digital Twin...
Initializing Virtual Asset: EV Traction Motor Digital Twin...
Initializing Virtual Asset: EV Traction Motor Digital Twin...
Initializing Virtual Asset: EV Traction Motor Digital Twin...
Initializing Virtual Asset: EV Traction Motor Digital Twin...
Initializing Virtual Asset: EV Traction Motor Digital Twin...
Initializing Virtual Asset: EV Traction Motor Digital Twin...
Initializing Virtual Asset: EV Traction Motor Digital Twin...
Initializing Vir

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore') # Hides annoying sci-kit warnings

print("EV Traction Motor Thermal Predictive Engine Active\n")

# 1. Load Data generated by the Digital Twin
try:
    print("Loading synthetic physics data from the Digital Twin...")
    powertrain_telemetry = pd.read_csv('synthetic_motor_data.csv')
    print("Data loaded successfully! Training AI brain...\n")
except FileNotFoundError:
    print("ERROR: I can't find 'synthetic_motor_data.csv'!")
    exit()

# 2. Split Data Channels into Inputs (X) and Target (y)
X = powertrain_telemetry[['Stator_Current_Amps', 'Coolant_Flow_Lmin']]
y = powertrain_telemetry['Turtle_Trigger']

# We use a smaller test_size here since our twin only generated 10 rows!
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=7)

# 3. Train the Predictive VCU Brain
ai_brain = LogisticRegression()
ai_brain.fit(X_train, y_train)

# Validate performance
predictions = ai_brain.predict(X_test)
score = accuracy_score(y_test, predictions)

print(f"VCU Core Training complete. Algorithm Prediction Accuracy: {score * 100:.1f}%\n")
print("-" * 65)
print("LIVE COCKPIT DIAGNOSTIC INTERFACE (EMULATING VEHICLE CAN BUS):")
print("-" * 65)

# 4. Interactive Live Prediction Interface
try:
    user_amps = float(input("Enter Live Inverter Stator Current (e.g. 400): "))
    user_flow = float(input("Enter Live Cooling Flow Rate (e.g. 3.0): "))

    sensor_packet = pd.DataFrame([[user_amps, user_flow]], columns=['Stator_Current_Amps', 'Coolant_Flow_Lmin'])
    prediction = ai_brain.predict(sensor_packet)[0]

    print("\n" + "="*55)
    print("EV POWERTRAIN SYSTEM STATUS REPORT:")
    print("="*55)
    if prediction == 1:
        print("ALERT: CRITICAL STATOR THERMAL OVERLOAD DETECTED!")
        print("Action: Engaging [TURTLE MODE]")
    else:
        print("STATUS: [SMOOTH CRUISE]")
        print("Thermal management optimal. Continuous torque permitted.")
    print("="*55)

except ValueError:
    print("Error: Input mismatch. Please enter numerical values only.")

EV Traction Motor Thermal Predictive Engine Active

Loading synthetic physics data from the Digital Twin...
Data loaded successfully! Training AI brain...

VCU Core Training complete. Algorithm Prediction Accuracy: 90.0%

-----------------------------------------------------------------
LIVE COCKPIT DIAGNOSTIC INTERFACE (EMULATING VEHICLE CAN BUS):
-----------------------------------------------------------------
Enter Live Inverter Stator Current (e.g. 400): 200
Enter Live Cooling Flow Rate (e.g. 3.0): 0

EV POWERTRAIN SYSTEM STATUS REPORT:
STATUS: [SMOOTH CRUISE]
Thermal management optimal. Continuous torque permitted.
